# Handwritten Character Recognition - EMNIST Letters (A-Z)
This notebook walks through the full pipeline: data, preprocessing, CNN, training, evaluation and custom-image prediction. Run it from the `notebooks/` folder or the project root. Reusable code lives in `src/`, so the notebook, the training script and the Streamlit app all share the same logic.

## 1. Import Libraries
We add the project root to `sys.path` so the `src` package can be imported.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, visualize as viz
from src.preprocessing import load_dataset, prepare_arrays, split_train_val, preprocess_image_with_steps
from src.model import build_cnn, compile_model, make_callbacks, save_label_mapping
from src.evaluate import evaluate_model
from src.predict import CharacterRecognizer

DATASET = "emnist"
config.ensure_directories()
keras.utils.set_random_seed(config.SEED)   # reproducibility
print("TensorFlow", tf.__version__)

## 2. Load Dataset
EMNIST Letters is downloaded with the `emnist` package (or read from CSV files in `data/`). The loader also **fixes the EMNIST orientation** (images are stored transposed) and shifts labels from 1-26 to 0-25.

In [ ]:
x_train_full, y_train_full, x_test, y_test, class_names = load_dataset(DATASET)
print("Loaded", len(x_train_full), "training and", len(x_test), "test images")

## 3. Understand Dataset
Inspect shapes, pixel range, number of classes and the label mapping.

In [ ]:
print("Image shape       :", x_train_full.shape[1:])
print("Training samples  :", len(x_train_full))
print("Testing samples   :", len(x_test))
print("Pixel dtype/range :", x_train_full.dtype, x_train_full.min(), "-", x_train_full.max())
print("Number of classes :", len(class_names))
print("Label mapping     :", {i: c for i, c in enumerate(class_names)})
pd.Series(y_train_full).map(lambda i: class_names[i]).value_counts().sort_index().to_frame("train samples").T

## 4. Data Visualization
Random labelled samples, class balance and the pixel-intensity distribution. The last figure compares raw vs. orientation-corrected EMNIST images - the bottom row must look upright.

In [ ]:
viz.plot_samples(x_train_full, y_train_full, class_names, n=24, title="Random training samples"); plt.show()
viz.plot_class_distribution(y_train_full, class_names); plt.show()
viz.plot_pixel_histogram(x_train_full); plt.show()
viz.plot_orientation_check(x_train_full); plt.show()

## 5. Data Preprocessing
1. Normalise pixels from 0-255 to 0-1.
2. Add a channel axis for the CNN: `(N, 28, 28)` -> `(N, 28, 28, 1)`.
3. Labels stay as integer class ids (we use sparse categorical cross-entropy, equivalent to one-hot encoding).

For **real-world images** the pipeline in `src/preprocessing.py` converts to grayscale, inverts dark-on-light writing, removes background (Otsu), crops, keeps aspect ratio, centres and resizes to 28x28. Below we simulate a paper photo (dark ink on white, large, noisy) from a training image and watch every step.

In [ ]:
sample = x_train_full[0]
paper = cv2.resize(255 - sample, (280, 280), interpolation=cv2.INTER_CUBIC)          # dark ink on white
paper = np.clip(paper.astype(np.int16) + np.random.default_rng(0).integers(-12, 12, paper.shape), 0, 255).astype(np.uint8)
processed, steps = preprocess_image_with_steps(paper)
viz.plot_preprocessing_steps({"Original training image": sample, "Simulated paper photo": paper, **steps}); plt.show()
print("Final CNN input:", processed.shape, processed.dtype, "range", processed.min(), "-", processed.max())

## 6. Train/Validation Split
A stratified 90/10 split keeps class proportions equal. The test set is never used for training or model selection.

In [ ]:
x_train, x_val, y_train, y_val = split_train_val(x_train_full, y_train_full)
x_train, x_val, x_test_n = prepare_arrays(x_train), prepare_arrays(x_val), prepare_arrays(x_test)
print("train", x_train.shape, "| validation", x_val.shape, "| test", x_test_n.shape)

## 7. CNN Architecture
Three convolution stages (with BatchNorm, ReLU, max-pooling and dropout) learn edges -> strokes -> shapes, followed by a small dense classifier with softmax output. Light augmentation (rotation, shift, zoom) is built into the model and is active only during training.

In [ ]:
model = build_cnn(len(class_names), augment=True, name=f"{DATASET}_cnn")
model.summary()

## 8. Model Compilation
Adam optimiser (lr = 1e-3), sparse categorical cross-entropy loss, accuracy metric.

In [ ]:
compile_model(model, learning_rate=1e-3)

## 9. Model Training
`ModelCheckpoint` keeps the **best** weights (lowest validation loss), `EarlyStopping` stops when validation loss stops improving and `ReduceLROnPlateau` lowers the learning rate. Each epoch prints training/validation loss and accuracy.

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30, batch_size=128, verbose=2,
    callbacks=make_callbacks(config.model_path(DATASET), config.history_path(DATASET)),
)

## 10. Training Curves

In [ ]:
viz.plot_history(history.history, f"{DATASET.upper()} training history"); plt.show()

## 11. Model Evaluation
Evaluate the best model on the unseen test set: accuracy, loss, precision, recall and F1 (macro and weighted). Everything is also saved under `outputs/`.

In [ ]:
best_model = keras.models.load_model(config.model_path(DATASET))
results = evaluate_model(DATASET, best_model, x_test_n, y_test, class_names,
                         extra={"num_train_samples": int(len(x_train)), "epochs_run": len(history.history["loss"])})
pd.Series(results["metrics"]).to_frame("value")

## 12. Confusion Matrix
Rows are actual classes, columns are predicted classes. Off-diagonal cells show which characters the model confuses.

In [ ]:
viz.plot_confusion_matrix(results["confusion_matrix"], class_names, title=f"{DATASET.upper()} confusion matrix"); plt.show()

## 13. Classification Report
Per-class precision, recall, F1-score and support.

In [ ]:
print(results["report_text"])

## 14. Correct/Incorrect Prediction Analysis
The misclassified samples shown are the model's **most confident mistakes**, which reveal its limits (e.g. look-alike strokes).

In [ ]:
y_pred, conf = results["y_pred"], results["confidence"]
correct = np.flatnonzero(y_test == y_pred)
wrong = np.flatnonzero(y_test != y_pred)
rng = np.random.default_rng(config.SEED)
viz.plot_prediction_grid(x_test_n, y_test, y_pred, conf, rng.choice(correct, 12, replace=False), class_names, "Correct predictions"); plt.show()
worst = wrong[np.argsort(-conf[wrong])][:12]
viz.plot_prediction_grid(x_test_n, y_test, y_pred, conf, worst, class_names, "Most confident mistakes"); plt.show()
print(f"{len(wrong)} of {len(y_test)} test images misclassified")

## 15. Custom Image Prediction
Put your own image at `sample_images/my_character.png` (any PNG/JPG of one handwritten character). If it does not exist, the cell uses a simulated paper photo built from a test image so the cell always runs.

In [ ]:
recognizer = CharacterRecognizer(best_model, class_names)
custom_path = PROJECT_ROOT / "sample_images" / "my_character.png"
if custom_path.exists():
    image = Image.open(custom_path)
    print("Using", custom_path)
else:
    demo = cv2.resize(255 - (x_test_n[0, ..., 0] * 255).astype(np.uint8), (280, 280), interpolation=cv2.INTER_CUBIC)
    image = Image.fromarray(demo)
    print("No custom image found; using a simulated photo of a test sample. Actual label:", class_names[y_test[0]])

result = recognizer.predict(image, top_k=3)
print(f"Prediction: {result.label}   Confidence: {result.confidence * 100:.1f}%")
for label, p in result.top_k:
    print(f"  {label} - {p * 100:.1f}%")
viz.plot_preprocessing_steps(result.steps, "Preprocessing of the custom image"); plt.show()

## 16. Save Model
The best checkpoint is already stored in `models/`. We also save the class-index -> character mapping used by the app.

In [ ]:
best_model.save(config.model_path(DATASET))
save_label_mapping(DATASET, class_names)
print("Saved:", config.model_path(DATASET).relative_to(PROJECT_ROOT), "and", config.labels_path(DATASET).relative_to(PROJECT_ROOT))

## 17. Final Results and Conclusion

In [ ]:
m = results["metrics"]
print(f"Dataset          : {DATASET.upper()}  ({len(class_names)} classes)")
print(f"Test accuracy    : {m['test_accuracy'] * 100:.2f}%")
print(f"Test loss        : {m['test_loss']:.4f}")
print(f"Macro precision  : {m['precision_macro']:.4f}")
print(f"Macro recall     : {m['recall_macro']:.4f}")
print(f"Macro F1-score   : {m['f1_macro']:.4f}")

**Conclusion.** A compact CNN with BatchNorm, dropout and light augmentation learns stroke-level features directly from pixels. The numbers above are measured on the held-out test set. Because EMNIST Letters merges upper- and lower-case forms into one class, look-alike pairs (e.g. I/L, Q/G, U/V) are the main source of errors. The same preprocessing is used for training data and uploaded/drawn images, and the convolutional backbone (`src/model.py: build_backbone`) can be reused as the feature extractor of a CRNN (CNN -> BiLSTM -> CTC) for full-word recognition.